In [2]:
from visualization_utils import TruthData
import torch
from plotly.subplots import make_subplots
import configparser

model = 'llama-2-13b'
config = configparser.ConfigParser()
config.read('config.ini')
layer = 14
noperiod = eval(config[model]['noperiod'])

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

In [16]:
TruthData.from_datasets(
    ['single-template/companion_farmed/rest_01', 'single-template/companion_wild/rest_01'], # datasets to use
    model='llama-2-13b',
    layer=layer,
    center=True,
    noperiod=noperiod,
    device=device
).plot(
    dimensions=2, # 3 dimensions also supported
    dim_offset=0, # increase if you want to ignore the first few PCs
    color='label',
    # if you don't want to plot and do PCA on all datasets:
    # plot_datasets = [ list of datasets to plot here ],
    # pca_datasets = [ list of datasets to use for PCA here]
)

In [5]:
# visualization of all single-template datasets
# one row per contrast pair, one column per template. Each panel gets its own
# PCA (pca_datasets = [dataset]), so within a panel the only thing that varies
# is the subject word -- no between-template variance to swamp the contrast.
pairs = [
    'human_animal',
    'human_companion',
    'human_farmed',
    'human_wild',
    'companion_farmed',
    'companion_wild',
    'farmed_wild',
]
templates = ['fire_01', 'flood_01', 'morning_01', 'rest_01']  # 2 harm, 2 neutral

datasets = [f'single-template/{pair}/{template}' for pair in pairs for template in templates]
n_rows, n_cols = len(pairs), len(templates)

td = TruthData.from_datasets(
    datasets,
    model=model,
    layer=layer,
    center=True,
    noperiod=noperiod,
    device=device)

fig = make_subplots(rows=n_rows, cols=n_cols,
                    column_titles=templates, row_titles=pairs,
                    horizontal_spacing=0.03, vertical_spacing=0.03)
for i, dataset in enumerate(datasets):
    for data in td.plot(
        dimensions=2,
        color='label',
        plot_datasets = [dataset],
        pca_datasets = [dataset],
        ).data:
        fig.add_trace(
            data, row=(i // n_cols) + 1, col=(i % n_cols) + 1
        )

for row in range(n_rows):
    for col in range(n_cols):
        fig.update_yaxes(
            scaleanchor = f"x{row * n_cols + col + 1}",
            scaleratio = 1,
            row=row+1,
            col=col+1,
        )

fig.update_coloraxes(
    colorscale = 'Bluered_r'
)

# 80 points per panel, so drop the tick clutter and let the shapes carry it
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)

fig.update_layout(
    height=220 * n_rows,
    width=220 * n_cols + 200,
    showlegend=False,
    title = {
        'text' : 'PCA of single-template datasets (label 1 = first category in the pair)',
        'font' : {'size' : 22}
    },
    coloraxis_showscale=False,
)

In [6]:
# does a direction found in one dataset separate the others?
# row = dataset the PCA basis comes from, col = dataset projected into it.
# The diagonal repeats the self-PCA above; the off-diagonal panels are the test.
pca_datasets = [
    'single-template/human_farmed/fire_01',       # harm basis
    'single-template/human_farmed/morning_01',    # neutral basis
    'single-template/human_animal/flood_01',      # broad human-vs-animal basis
]

plot_datasets = [
    # same contrast, different templates -> does the direction survive a template change?
    'single-template/human_farmed/fire_01',
    'single-template/human_farmed/flood_01',
    'single-template/human_farmed/morning_01',
    'single-template/human_farmed/rest_01',
    # different contrasts -> is it really human vs animal?
    'single-template/human_animal/flood_01',
    'single-template/human_wild/flood_01',
    # animal vs animal: a human/animal direction should NOT separate these
    'single-template/companion_farmed/flood_01',
    'single-template/farmed_wild/flood_01',
]

# every pca_dataset must also be plotted, since td is built from plot_datasets
assert set(pca_datasets) <= set(plot_datasets), set(pca_datasets) - set(plot_datasets)

fig = make_subplots(rows=len(pca_datasets), cols=len(plot_datasets), 
    vertical_spacing=0.05, horizontal_spacing=0.01,
    )

for col, plot_dataset in enumerate(plot_datasets):
    fig.update_xaxes(title= {
                    'text': plot_dataset.replace('/', '<br>'),
                    'font' : {'size' : 11}
                    },
                    row=len(pca_datasets), col=col+1
                )
for row, pca_dataset in enumerate(pca_datasets):
    fig.update_yaxes(title= {
                    'text': pca_dataset.replace('/', '<br>'),
                    'font' : {'size' : 11}
                    },
                    row=row+1, col=1
                )

td = TruthData.from_datasets(plot_datasets, model=model, layer=layer, center=True, noperiod=noperiod, device=device)

for row, pca_dataset in enumerate(pca_datasets):
    for col, plot_dataset in enumerate(plot_datasets):
        subfig = td.plot(
            dimensions = 2,
            plot_datasets = [plot_dataset],
            pca_datasets = [pca_dataset],
            color='label',
        )
        fig.add_trace(subfig.data[0], row=row+1, col=col+1)

for row in range(len(pca_datasets)):
    for col in range(len(plot_datasets)):
        fig.update_yaxes(
            scaleanchor = f"x{row * len(plot_datasets) + col + 1}",
            scaleratio = 1,
            row=row+1,
            col=col+1,
        )

fig.update_coloraxes(
    colorscale = 'Bluered_r'
)

fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)

fig.update_layout(
    height=700,
    width=1600,
    coloraxis_showscale=False,
    title = {
        'text': f'Single-template datasets in various PCA bases (row = basis, col = plotted)',
        'font' : {'size' : 24}
    }
)

fig.show()

In [11]:
from plotly.subplots import make_subplots
import numpy as np

# Does the human/animal split keep the same orientation under harm and neutral
# scenarios? Each panel is a joint PCA over one contrast's harm + neutral
# template, with four colors so the two axes can be told apart.
pairs = [
    ['single-template/human_farmed/flood_01', 'single-template/human_farmed/morning_01'],
    ['single-template/human_animal/flood_01', 'single-template/human_animal/morning_01'],
    ['single-template/farmed_wild/flood_01', 'single-template/farmed_wild/morning_01'],  # control: animal vs animal
]
layer = 12

# colormappings under the Rainbow colorscale, with cmin/cmax pinned to [0, 1]
# below so these stay the same hues regardless of what's in each panel
RED = 1
BLUE = .17
PURPLE = 0
YELLOW = .73

# harm:    label 1 (first category) = BLUE,   label 0 (second) = RED
# neutral: label 1 (first category) = YELLOW, label 0 (second) = PURPLE
# BLUE and YELLOW should land on the same side if the direction is shared.

fig = make_subplots(rows=1, cols=len(pairs),
                    shared_yaxes=True,
                    x_title='PC1', y_title='PC2',
                    subplot_titles=[ pair[0].split('/')[0] for pair in pairs ]
                    )

for i, pair in enumerate(pairs):
    td = TruthData.from_datasets(
        pair,
        model=model,
        layer=layer,
        noperiod=noperiod,
        device=device
    )

    # write the color into its own float column. Assigning via
    # td.df.loc[pair[0], 'label'] would align an inner-level-indexed Series
    # against the full MultiIndex, silently producing all-NaN (-> black points).
    dataset = td.df.index.get_level_values(0)
    td.df['color'] = np.where(
        dataset == pair[0],
        np.where(td.df['label'] == 1, BLUE, RED),       # harm
        np.where(td.df['label'] == 1, YELLOW, PURPLE),  # neutral
    )

    subfig = td.plot(
        dimensions=2,
        color = 'color',
    )

    fig.add_trace(subfig.data[0], row=1, col=i+1)

fig.update_coloraxes(
    colorscale='Rainbow',
    cmin=0,
    cmax=1,
)

fig.update_layout(
    height=450,
    width=1300,
    coloraxis_showscale=False,
    title = {
        'text' : 'harm vs neutral: blue/red = harm, yellow/purple = neutral',
        'font' : {'size' : 20}
    }
)

fig.show()

In [9]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import os

# where does the human/animal split emerge across depth?
datasets = [
    'single-template/human_farmed/flood_01',      # harm
    'single-template/human_farmed/morning_01',    # neutral
    'single-template/farmed_wild/flood_01',       # control: animal vs animal
]
layers = [4, 8, 14, 22, 36]  # llama-2-13b has 40 layers; 14 is its probe_layer

figs = [[] for _ in datasets]

for i, dataset in enumerate(datasets):
    for sweep_layer in layers:
        fig = TruthData.from_datasets(
            [dataset],
            model=model,
            layer=sweep_layer,
            noperiod=noperiod,
            device=device
            ).plot(
                dimensions=2,
                color='label',
            )
        figs[i].append(fig)

fig = make_subplots(rows = len(datasets), cols = len(layers),
                    subplot_titles=[f"layer {sweep_layer}" for sweep_layer in layers],
                    vertical_spacing=0.05)

for i, dataset in enumerate(datasets):
    for j, sweep_layer in enumerate(layers):
        for data in figs[i][j].data:
            data['showlegend'] = False
            fig.add_trace(data, row=i+1, col=j+1)

for i, dataset in enumerate(datasets):
    fig.update_yaxes(title_text=dataset.replace('/', '<br>'), title_font={'size': 11}, row=i+1, col=1)

fig.update_coloraxes(
    colorscale='Bluered_r'
)

fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)

fig.update_layout(height=800, width=1200, coloraxis_showscale=False)

fig.update_layout(
    title = {
        'text' : f"Single-template datasets across layers",
        'font' : {'size' : 20},
    }
)


fig.show()